<a href="https://colab.research.google.com/github/HithaBadikillaya/Speech-Emotion-Recognition/blob/main/notebooks/wavlm_ce_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WavLM + Cross-Entropy Baseline Training
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
# EDIT these paths to match where you stored your data on Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/SER_data'
SAVE_TO_DRIVE   = '/content/drive/MyDrive/SER_runs'
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

In [ ]:
!git clone https://github.com/HithaBadikillaya/Speech-Emotion-Recognition.git
%cd Speech-Emotion-Recognition
!git log --oneline -3

In [ ]:
!pip install -q transformers soundfile scipy scikit-learn tensorboard seaborn
print('Done!')

In [ ]:
import os
# Symlink your Drive data folder into the repo's data/ directory
if os.path.exists(DRIVE_DATA_PATH) and not os.path.exists('data'):
    os.symlink(DRIVE_DATA_PATH, 'data')
    print('Linked data from Drive')
elif not os.path.exists(DRIVE_DATA_PATH):
    print(f'WARNING: {DRIVE_DATA_PATH} not found! Edit DRIVE_DATA_PATH above.')
!echo 'Train:';  wc -l data/metadata/train.csv
!echo 'Val:';    wc -l data/metadata/validation.csv

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Sanity check — should complete in ~30 seconds on GPU
!PYTHONPATH=. python src/training/train.py --config configs/wavlm_ce.yaml --dry-run

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

In [ ]:
# Full training — ~1-2 hours on T4 GPU
!PYTHONPATH=. python src/training/train.py --config configs/wavlm_ce.yaml

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
with open('runs/wavlm_ce/results.json') as f:
    r = json.load(f)
print(f"Best UAR  : {r['best_uar']:.4f}")
print(f"Final UAR : {r['final_val_uar']:.4f}")
print(f"WAR (Acc) : {r['final_val_war']:.4f}")
print(f"F1 Macro  : {r['final_val_f1_macro']:.4f}")

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
cm = np.array(r['confusion_matrix'])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f"WavLM CE — Confusion Matrix (UAR={r['best_uar']:.3f})")
plt.tight_layout()
plt.savefig('runs/wavlm_ce/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import shutil
dest = os.path.join(SAVE_TO_DRIVE, 'wavlm_ce')
shutil.copytree('runs/wavlm_ce', dest, dirs_exist_ok=True)
print(f'Saved to Drive: {dest}')
!ls {dest}

WavLM Standard SupCon Project

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!nvidia-smi

Sun Sep 20 06:38:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!git clone -b feature/ser-project https://github.com/Deepthi055/Speech-Emotion-Recognition.git

Cloning into 'Speech-Emotion-Recognition'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 117 (delta 37), reused 100 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 32.59 KiB | 4.66 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [7]:
%cd Speech-Emotion-Recognition

/content/Speech-Emotion-Recognition


In [8]:
!git branch --show-current

feature/ser-project


In [9]:
!ls

configs  notebooks  README.md  requirements.txt  src


In [10]:
!git branch --show-current
!ls

feature/ser-project
configs  notebooks  README.md  requirements.txt  src


In [11]:
!ls data/metadata

ls: cannot access 'data/metadata': No such file or directory


In [12]:
!find data -type f | head -30

find: ‘data’: No such file or directory


In [13]:
import pandas as pd

train_df = pd.read_csv("data/metadata/train.csv")

print(train_df.head())
print("Columns:", train_df.columns.tolist())
print("Number of samples:", len(train_df))

FileNotFoundError: [Errno 2] No such file or directory: 'data/metadata/train.csv'

In [14]:
!git pull origin feature/ser-project


remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 30 (delta 12), reused 28 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (30/30), 10.22 KiB | 1.28 MiB/s, done.
From https://github.com/Deepthi055/Speech-Emotion-Recognition
 * branch            feature/ser-project -> FETCH_HEAD
   b0fb299..883521d  feature/ser-project -> origin/feature/ser-project
Updating b0fb299..883521d
Fast-forward
 configs/wavlm_ce_baseline.yaml    |  27 +++++
 scripts/extract_embeddings.py     | 122 +++++++++++++++++++++++
 scripts/prepare_dataset.py        | 148 ++++++++++++++++++++++++++++
 scripts/train_ce.py               | 201 ++++++++++++++++++++++++++++++++++++++
 src/datasets/embedding_dataset.py |  35 +++++++
 src/datasets/emobox_dataset.py    |  11 ++-
 src/models/baseline.py            |  11 +++
 7 files changed, 550 insertions(+), 5 deletions(-)
 create mode 100644 configs/wavlm_ce_baseline.yam

In [15]:
!python -m py_compile src/datasets/emobox_dataset.py

In [19]:
!find "/content/drive/MyDrive" -type f -name "train.csv"

^C


In [22]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [24]:
import os

print(os.listdir("/content/drive/MyDrive"))

['2024-01-20-file-2.sagews.html', '2024-01-20-file-2.sagews.gdoc', 'adhar card 1.pdf', 'Deepthi  (3).pdf', 'Deepthi .pdf', 'ETC Assignment 6.docx', '1st year engineering ', 'VTU fest chikkaballapura', 'VTU fest 2', '1st year2', 'HH-07.pdf', 'Colab Notebooks', 'Yoga, Tumkur', 'Deepthi (1).pdf', 'marks card 1st year.pdf', 'ration card (2) (1).pdf', 'ration card (2).pdf', 'PDFGallery_20241011_204219.pdf', 'html assignment', 'Prepare the Planting Hole1 (2).pptx', 'Prepare the Planting Hole1 (1).pptx', 'Prepare the Planting Hole1.pptx', 'package.json', 'Cart (1).js', 'Assignment 2', 'App.js', 'CakeDetail.js', 'Cart.js', 'About.js', 'Home.js', 'Contact.js', 'cakecatogaries.js', 'CakeList.js', 'HomePage.js', 'Navbar.js', 'Slider.js', 'Recording 2024-12-24 211900.mp4', 'star hotel final report.pdf', 'DEEPTHI (4SF23CS055).pdf', 'DocScanner Mar 11, 2025 8-36 PM.pdf', 'Tables and its basic concept.pdf', 'Untitled document (4).gdoc', 'Screenshot 2025-03-29 180124.png', 'Calculator Project Report.p

In [25]:
import os

print(os.listdir("/content/drive/MyDrive"))

['2024-01-20-file-2.sagews.html', '2024-01-20-file-2.sagews.gdoc', 'adhar card 1.pdf', 'Deepthi  (3).pdf', 'Deepthi .pdf', 'ETC Assignment 6.docx', '1st year engineering ', 'VTU fest chikkaballapura', 'VTU fest 2', '1st year2', 'HH-07.pdf', 'Colab Notebooks', 'Yoga, Tumkur', 'Deepthi (1).pdf', 'marks card 1st year.pdf', 'ration card (2) (1).pdf', 'ration card (2).pdf', 'PDFGallery_20241011_204219.pdf', 'html assignment', 'Prepare the Planting Hole1 (2).pptx', 'Prepare the Planting Hole1 (1).pptx', 'Prepare the Planting Hole1.pptx', 'package.json', 'Cart (1).js', 'Assignment 2', 'App.js', 'CakeDetail.js', 'Cart.js', 'About.js', 'Home.js', 'Contact.js', 'cakecatogaries.js', 'CakeList.js', 'HomePage.js', 'Navbar.js', 'Slider.js', 'Recording 2024-12-24 211900.mp4', 'star hotel final report.pdf', 'DEEPTHI (4SF23CS055).pdf', 'DocScanner Mar 11, 2025 8-36 PM.pdf', 'Tables and its basic concept.pdf', 'Untitled document (4).gdoc', 'Screenshot 2025-03-29 180124.png', 'Calculator Project Report.p

In [27]:
!find "/content/drive/MyDrive/data" -maxdepth 3 -type f | head -30

In [28]:
import os

data_path = "/content/drive/MyDrive/data"

print("Exists:", os.path.exists(data_path))
print("Is directory:", os.path.isdir(data_path))
print("Contents:", os.listdir(data_path))

Exists: True
Is directory: True
Contents: ['raw', 'metadata']


In [29]:
!find "/content/drive/MyDrive/data/metadata" -maxdepth 2 -type f

/content/drive/MyDrive/data/metadata/cremad_train.csv
/content/drive/MyDrive/data/metadata/cremad_test.csv
/content/drive/MyDrive/data/metadata/cremad_validation.csv
/content/drive/MyDrive/data/metadata/iemocap_test.csv
/content/drive/MyDrive/data/metadata/iemocap_train.csv
/content/drive/MyDrive/data/metadata/iemocap_validation.csv
/content/drive/MyDrive/data/metadata/test.csv
/content/drive/MyDrive/data/metadata/validation.csv
/content/drive/MyDrive/data/metadata/train.csv
/content/drive/MyDrive/data/metadata/train_combined.csv
/content/drive/MyDrive/data/metadata/validation_combined.csv
/content/drive/MyDrive/data/metadata/test_combined.csv
/content/drive/MyDrive/data/metadata/train_step5.csv
/content/drive/MyDrive/data/metadata/validation_step5.csv
/content/drive/MyDrive/data/metadata/test_step5.csv


In [30]:
%cd /content/Speech-Emotion-Recognition

/content/Speech-Emotion-Recognition


In [31]:
!cp -r "/content/drive/MyDrive/data" .

In [32]:
!ls data/metadata

cremad_test.csv        iemocap_validation.csv  train.csv
cremad_train.csv       test_combined.csv       train_step5.csv
cremad_validation.csv  test.csv		       validation_combined.csv
iemocap_test.csv       test_step5.csv	       validation.csv
iemocap_train.csv      train_combined.csv      validation_step5.csv


In [33]:
!find data/raw -type f | head -5

data/raw/CREMA-D/AudioWAV/1042_ITH_DIS_XX.wav
data/raw/CREMA-D/AudioWAV/1042_IEO_HAP_HI.wav
data/raw/CREMA-D/AudioWAV/1042_IOM_ANG_XX.wav
data/raw/CREMA-D/AudioWAV/1042_IEO_FEA_HI.wav
data/raw/CREMA-D/AudioWAV/1042_ITH_FEA_XX.wav


In [35]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
GPU name: Tesla T4


In [36]:
!python -m src.training.train --config configs/wavlm_supcon.yaml

2026-09-20 07:59:24.001646: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-20 07:59:42] [INFO] Starting experiment: wavlm_supcon (variant: standard_supcon)
[2026-09-20 07:59:42] [INFO] Using device: cuda
config.json: 100% 2.24k/2.24k [00:00<00:00, 1.20MB/s]

pytorch_model.bin: downloading bytes:  22% 81.6M/378M [00:02<00:03, 93.2MB/s, 4.60MB/s  ]
pytorch_model.bin: downloading bytes:  28% 107M/378M [00:02<00:02, 122MB/s, 7.68MB/s  ]  
pytorch_model.bin: downloading bytes:  40% 150M/378M [00:02<00:01, 118MB/s, 12.1MB/s  ]
pytorch_model.bin: reconstructing file:  53% 200M/378M [00:02<00:01, 103MB/s, 8.38MB/s  ]  
pytorch_model.bin: downloading bytes:  56% 213M/378M [00:03<00:01, 129MB/s, 17.1MB/s  ]
pytorch_model.bin: downloading bytes:  

In [37]:
import json

with open("runs/wavlm_supcon/results.json", "r") as f:
    results = json.load(f)

print(json.dumps(results, indent=2))

{
  "experiment": "wavlm_supcon",
  "variant": "standard_supcon",
  "best_uar": 0.16666666666666666,
  "final_val_uar": 0.16666666666666666,
  "final_val_war": 0.18181818181818182,
  "final_val_f1_macro": 0.05128205128205129,
  "final_val_f1_weighted": 0.055944055944055944,
  "confusion_matrix": [
    [
      0,
      40,
      0,
      0,
      0,
      0
    ],
    [
      0,
      40,
      0,
      0,
      0,
      0
    ],
    [
      0,
      40,
      0,
      0,
      0,
      0
    ],
    [
      0,
      40,
      0,
      0,
      0,
      0
    ],
    [
      0,
      20,
      0,
      0,
      0,
      0
    ],
    [
      0,
      40,
      0,
      0,
      0,
      0
    ]
  ]
}


In [39]:
import pandas as pd

train_df = pd.read_csv("data/metadata/train.csv")
val_df = pd.read_csv("data/metadata/validation.csv")

print("Train emotion distribution:")
print(train_df["emotion"].value_counts().sort_index())

print("\nValidation emotion distribution:")
print(val_df["emotion"].value_counts().sort_index())

print("\nEmotion labels:")
print(train_df["emotion"].unique())

Train emotion distribution:
emotion
angry      112
disgust    112
fear       112
happy      112
neutral     56
sad        112
Name: count, dtype: int64

Validation emotion distribution:
emotion
angry      40
disgust    40
fear       40
happy      40
neutral    20
sad        40
Name: count, dtype: int64

Emotion labels:
['neutral' 'happy' 'sad' 'angry' 'fear' 'disgust']


In [40]:
!grep -n -E "emotion|label|class" src/datasets/emobox_dataset.py | head -30

20:class SERDataset(Dataset):
66:        emotion_str = row["emotion"]
67:        label = EMOTION_MAP[emotion_str]
77:            "label": label,
82:            "emotion_str": emotion_str,
88:    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
105:        "labels": labels,


In [41]:
!sed -n '1,25p' src/datasets/emobox_dataset.py

import os
from math import gcd
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset
from scipy.signal import resample_poly

EMOTION_MAP = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "neutral": 4,
    "sad": 5,
}

class SERDataset(Dataset):
    def __init__(self, metadata_path_or_df, target_sr: int = 16000, max_seconds: float = 6.0, max_samples: int = None):
        super().__init__()
        if isinstance(metadata_path_or_df, (str, Path)):
            self.df = pd.read_csv(metadata_path_or_df)
        else:


In [42]:
from src.datasets.emobox_dataset import SERDataset
from collections import Counter

dataset = SERDataset(
    "data/metadata/train.csv",
    target_sr=16000,
    max_seconds=6.0
)

labels = [dataset[i]["label"] for i in range(len(dataset))]

print("Total samples:", len(labels))
print("Label distribution:", Counter(labels))
print("First 10 labels:", labels[:10])

Total samples: 616
Label distribution: Counter({3: 112, 5: 112, 0: 112, 2: 112, 1: 112, 4: 56})
First 10 labels: [4, 4, 4, 4, 3, 3, 3, 3, 3, 3]


In [43]:
!cat configs/wavlm_supcon.yaml

experiment:
  name: "wavlm_supcon"
  variant: "standard_supcon"

model:
  model_id: "microsoft/wavlm-base"
  num_classes: 6
  proj_dim: 128
  use_projection: true
  use_classifier: true
  pooling: "attention"
  freeze_backbone: true
data:
  train_csv: "data/metadata/train.csv"
  val_csv: "data/metadata/validation.csv"
  test_csv: "data/metadata/test.csv"
  target_sr: 16000
  max_seconds: 6.0

sampler:
  type: "standard"
  batch_size: 16
  shuffle: true

loss:
  temperature: 0.07

training:
  epochs: 10
  lr: 1e-4
  weight_decay: 1e-4
  supcon_weight: 1.0
  ce_weight: 1.0


In [44]:
!grep -n -A15 -B10 "supcon_loss\|ce_loss\|SupConLoss" src/training/train.py

4-import numpy as np
5-from pathlib import Path
6-import yaml
7-import torch
8-import torch.nn as nn
9-from torch.utils.data import DataLoader
10-from torch.utils.tensorboard import SummaryWriter
11-
12-from src.datasets.emobox_dataset import SERDataset, collate_fn
13-from src.models.ser_model import WavLMSupConModel
14:from src.losses.supervised_contrastive import SupConLoss
15-from src.samplers.standard import get_standard_sampler
16-from src.training.evaluate import evaluate
17-from src.utils.logging import setup_logger
18-from src.utils.seed import set_seed
19-
20-logger = setup_logger()
21-
22-
23-def load_config(config_path: str) -> dict:
24-    with open(config_path, "r") as f:
25-        return yaml.safe_load(f)
26-
27-
28-def build_dataloader(dataset, config_sampler, is_train: bool = True):
29-    sampler_type = config_sampler.get("type", "standard") if is_train else "standard"
--
63-            projections = outputs["projections"]
64-            if variant == "standard_supcon

In [45]:
!grep -n -A15 -B5 "freeze_backbone" src/models/ser_model.py

13-        num_classes: int = 6,
14-        proj_dim: int = 128,
15-        use_projection: bool = True,
16-        use_classifier: bool = True,
17-        pooling: str = "attention",
18:        freeze_backbone: bool = False,
19-    ):
20-        super().__init__()
21-
22-        self.backbone = WavLMBackbone(
23-            model_id=model_id,
24-            pooling=pooling
25-        )
26-
27:        if freeze_backbone:
28-            for param in self.backbone.model.parameters():
29-                param.requires_grad = False
30-
31-        self.projection_head = (
32-            ProjectionHead(
33-                in_dim=768,
34-                out_dim=proj_dim
35-            )
36-            if use_projection
37-            else None
38-        )
39-
40-        self.classifier_head = (
41-            ClassifierHead(
42-                in_dim=768,


In [46]:
from pathlib import Path

path = Path("src/training/train.py")
text = path.read_text()

old = '''        pooling=model_cfg.get("pooling", "attention"),
    ).to(device)'''

new = '''        pooling=model_cfg.get("pooling", "attention"),
        freeze_backbone=model_cfg.get("freeze_backbone", False),
    ).to(device)'''

if old in text:
    text = text.replace(old, new)
    path.write_text(text)
    print("Fix applied successfully.")
else:
    print("Target code not found. Check the model initialization manually.")

Fix applied successfully.


In [47]:
!grep -n -A10 "model = WavLMSupConModel" src/training/train.py

136:    model = WavLMSupConModel(
137-        model_id=model_cfg.get("model_id", "microsoft/wavlm-base"),
138-        num_classes=model_cfg.get("num_classes", 6),
139-        proj_dim=model_cfg.get("proj_dim", 128),
140-        use_projection=model_cfg.get("use_projection", True),
141-        use_classifier=model_cfg.get("use_classifier", True),
142-        pooling=model_cfg.get("pooling", "attention"),
143-        freeze_backbone=model_cfg.get("freeze_backbone", False),
144-    ).to(device)
145-
146-    ce_criterion = nn.CrossEntropyLoss()


In [48]:
from src.models.ser_model import WavLMSupConModel

model = WavLMSupConModel(
    model_id="microsoft/wavlm-base",
    num_classes=6,
    proj_dim=128,
    use_projection=True,
    use_classifier=True,
    pooling="attention",
    freeze_backbone=True,
)

frozen = 0
trainable = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable += param.numel()
    else:
        frozen += param.numel()

print("Frozen parameters:", frozen)
print("Trainable parameters:", trainable)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Frozen parameters: 94381936
Trainable parameters: 530567


In [49]:
!git diff -- src/training/train.py

diff --git a/src/training/train.py b/src/training/train.py
index b8f551c..8e41b0c 100644
--- a/src/training/train.py
+++ b/src/training/train.py
@@ -140,6 +140,7 @@ def main():
         use_projection=model_cfg.get("use_projection", True),
         use_classifier=model_cfg.get("use_classifier", True),
         pooling=model_cfg.get("pooling", "attention"),
+        freeze_backbone=model_cfg.get("freeze_backbone", False),
     ).to(device)
 
     ce_criterion = nn.CrossEntropyLoss()


In [50]:
import os
import json
import subprocess
import yaml
import torch

print("=" * 50)
print("TASK 7 FINAL VERIFICATION")
print("=" * 50)

# 1. Check Git branch and latest commit
print("\n[1] Git verification")
print(subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())

print(subprocess.check_output(
    ["git", "log", "-1", "--oneline"], text=True
).strip())

# 2. Check configuration
print("\n[2] Configuration verification")

with open("configs/wavlm_supcon.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Variant:", config["experiment"]["variant"])
print("Projection enabled:", config["model"]["use_projection"])
print("Classifier enabled:", config["model"]["use_classifier"])
print("Backbone frozen:", config["model"]["freeze_backbone"])
print("Sampler:", config["sampler"]["type"])
print("Temperature:", config["loss"]["temperature"])

# 3. Check saved results
print("\n[3] Results verification")

results_path = "runs/wavlm_supcon/results.json"

if os.path.exists(results_path):
    with open(results_path, "r") as f:
        results = json.load(f)

    print("Results file: EXISTS")
    print("Best UAR:", results["best_uar"])
    print("Final UAR:", results["final_val_uar"])
    print("Final WAR:", results["final_val_war"])
    print("Macro-F1:", results["final_val_f1_macro"])
    print("Confusion matrix:")

    for row in results["confusion_matrix"]:
        print(row)
else:
    print("Results file: NOT FOUND")

# 4. Check implementation files
print("\n[4] Implementation files")

required_files = [
    "configs/wavlm_supcon.yaml",
    "src/losses/supervised_contrastive.py",
    "src/models/projection.py",
    "src/models/ser_model.py",
    "src/training/train.py",
    "src/training/evaluate.py",
    "src/utils/metrics.py",
]

for file in required_files:
    print(f"{file}: {'EXISTS' if os.path.exists(file) else 'MISSING'}")

# 5. Check GPU
print("\n[5] GPU verification")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n" + "=" * 50)
print("VERIFICATION COMPLETED")
print("=" * 50)

TASK 7 FINAL VERIFICATION

[1] Git verification
feature/ser-project
883521d Merge branch 'main' into feature/ser-project

[2] Configuration verification
Variant: standard_supcon
Projection enabled: True
Classifier enabled: True
Backbone frozen: True
Sampler: standard
Temperature: 0.07

[3] Results verification
Results file: EXISTS
Best UAR: 0.16666666666666666
Final UAR: 0.16666666666666666
Final WAR: 0.18181818181818182
Macro-F1: 0.05128205128205129
Confusion matrix:
[0, 40, 0, 0, 0, 0]
[0, 40, 0, 0, 0, 0]
[0, 40, 0, 0, 0, 0]
[0, 40, 0, 0, 0, 0]
[0, 20, 0, 0, 0, 0]
[0, 40, 0, 0, 0, 0]

[4] Implementation files
configs/wavlm_supcon.yaml: EXISTS
src/losses/supervised_contrastive.py: EXISTS
src/models/projection.py: EXISTS
src/models/ser_model.py: EXISTS
src/training/train.py: EXISTS
src/training/evaluate.py: EXISTS
src/utils/metrics.py: EXISTS

[5] GPU verification
CUDA available: True
GPU: Tesla T4

VERIFICATION COMPLETED


In [51]:
!git status
!git log --oneline -5
!git show --stat --oneline HEAD

On branch feature/ser-project
Your branch is up to date with 'origin/feature/ser-project'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/training/train.py

no changes added to commit (use "git add" and/or "git commit -a")
883521d (HEAD -> feature/ser-project, origin/feature/ser-project, origin/HEAD) Merge branch 'main' into feature/ser-project
b0fb299 Implement standard supervised contrastive learning
6712d96 feat(model): implement CPU-only WavLM + Cross-Entropy baseline classifier
f056728 feat(embeddings): add frozen WavLM CPU feature extractor with mean pooling
374a163 feat(data): add unified dataset preparation and speaker-independent splits
883521d (HEAD -> feature/ser-project, origin/feature/ser-project, origin/HEAD) Merge branch 'main' into feature/ser-project

 configs/wavlm_ce_baseline.yaml    |  27 +++++
 scripts/extract_embeddings.py     | 12

In [52]:
!git diff -- src/training/train.py

diff --git a/src/training/train.py b/src/training/train.py
index b8f551c..8e41b0c 100644
--- a/src/training/train.py
+++ b/src/training/train.py
@@ -140,6 +140,7 @@ def main():
         use_projection=model_cfg.get("use_projection", True),
         use_classifier=model_cfg.get("use_classifier", True),
         pooling=model_cfg.get("pooling", "attention"),
+        freeze_backbone=model_cfg.get("freeze_backbone", False),
     ).to(device)
 
     ce_criterion = nn.CrossEntropyLoss()


In [56]:
from pathlib import Path

readme_path = Path("README.md")

print(readme_path.read_text())

In [57]:
from pathlib import Path

readme_path = Path("README.md")

print("README exists:", readme_path.exists())
print("README size:", readme_path.stat().st_size if readme_path.exists() else 0, "bytes")
print("\nREADME content:\n")
print(readme_path.read_text() if readme_path.exists() else "README not found")

README exists: True
README size: 0 bytes

README content:




In [58]:
from pathlib import Path

readme_content = r"""# Speech Emotion Recognition

## Task 7: Standard Supervised Contrastive Learning

### Objective

To implement Standard Supervised Contrastive Learning (SupCon) for Speech Emotion Recognition using WavLM embeddings.

### Implementation

The following components were implemented:

1. Added a projection head after the WavLM feature extraction layer.
2. Implemented Standard Supervised Contrastive Loss.
3. Created positive pairs using samples belonging to the same emotion class.
4. Added a standard batch sampler.
5. Integrated supervised contrastive loss with the classification loss.
6. Added support for freezing the WavLM backbone.
7. Implemented evaluation using Accuracy, UAR, Macro-F1, and a confusion matrix.
8. Kept the WavLM + Cross-Entropy baseline configuration separate.

### Model Configuration

| Parameter | Value |
|---|---|
| Backbone | microsoft/wavlm-base |
| Learning approach | Standard SupCon |
| Projection head | Enabled |
| Classifier | Enabled |
| Projection dimension | 128 |
| Pooling | Attention |
| Backbone freezing | Enabled in configuration |
| Batch sampler | Standard |
| Temperature | 0.07 |
| Learning rate | 1e-4 |
| Training epochs | 10 |

### Initial Training Results

An initial training run was performed using the Standard SupCon configuration.

| Metric | Result |
|---|---:|
| Best UAR | 16.67% |
| Final UAR | 16.67% |
| Final Weighted Accuracy | 18.18% |
| Macro-F1 | 5.13% |

### Initial Confusion Matrix

The initial validation results showed that the model predicted the same emotion class for all validation samples.

| Actual Emotion | Predicted Emotion |
|---|---|
| Angry | Disgust |
| Disgust | Disgust |
| Fear | Disgust |
| Happy | Disgust |
| Neutral | Disgust |
| Sad | Disgust |

These results indicate that the initial training run did not achieve effective emotion classification.

The results are documented as initial training results and should not be interpreted as evidence of successful model learning.

### Limitations and Future Work

- Investigate why predictions were concentrated in a single emotion class.
- Verify training behavior after applying the backbone-freezing configuration fix.
- Perform another training run if final performance evaluation of the corrected implementation is required.
- Analyze class-wise performance and the confusion matrix.

### Status

The Standard SupCon implementation and initial evaluation were completed. The initial model performance requires further investigation.
"""

readme_path = Path("README.md")
readme_path.write_text(readme_content, encoding="utf-8")

print("README.md updated successfully.")
print("File size:", readme_path.stat().st_size, "bytes")

README.md updated successfully.
File size: 2481 bytes


In [59]:
from pathlib import Path

print(Path("README.md").read_text(encoding="utf-8"))

# Speech Emotion Recognition

## Task 7: Standard Supervised Contrastive Learning

### Objective

To implement Standard Supervised Contrastive Learning (SupCon) for Speech Emotion Recognition using WavLM embeddings.

### Implementation

The following components were implemented:

1. Added a projection head after the WavLM feature extraction layer.
2. Implemented Standard Supervised Contrastive Loss.
3. Created positive pairs using samples belonging to the same emotion class.
4. Added a standard batch sampler.
5. Integrated supervised contrastive loss with the classification loss.
6. Added support for freezing the WavLM backbone.
7. Implemented evaluation using Accuracy, UAR, Macro-F1, and a confusion matrix.
8. Kept the WavLM + Cross-Entropy baseline configuration separate.

### Model Configuration

| Parameter | Value |
|---|---|
| Backbone | microsoft/wavlm-base |
| Learning approach | Standard SupCon |
| Projection head | Enabled |
| Classifier | Enabled |
| Projection dimension | 1

In [62]:
!git status


On branch feature/ser-project
Your branch is up to date with 'origin/feature/ser-project'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   README.md
	modified:   src/training/train.py



In [61]:
!git add README.md src/training/train.py

In [67]:
!git config --global user.name "Deepthi055"
!git config --global user.email "deepthiacharya2005@gmail.com"

In [68]:
!git commit -m "Implement Standard SupCon and document Task 7"

[feature/ser-project d96d627] Implement Standard SupCon and document Task 7
 2 files changed, 76 insertions(+)


In [69]:
!git push origin feature/ser-project

fatal: could not read Username for 'https://github.com': No such device or address


In [70]:
!sudo apt-get update -qq
!sudo apt-get install gh -y -qq

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../archives/gh_2.101.0_amd64.deb ...
Unpacking gh (

In [ ]:
!gh auth login

78? Where do you use GitHub?  [Use arrows to move, type to filter]
> GitHub.com
  Other
788? Where do you use GitHub? GitHub.com
78? What is your preferred protocol for Git operations on this host?  [Use arrows to move, type to filter]
> HTTPS
  SSH
788? What is your preferred protocol for Git operations on this host? HTTPS
? Authenticate Git with your GitHub credentials? (Y/n) 7